In [1]:
# Diffusion model dependencies (TabDDPM + ForestDiffusion)
# TabDDPM: yandex-research/tab-ddpm (_vendor/tab-ddpm)
# ForestDiffusion: pip install ForestDiffusion
# libzero/rtdl pin torch<2; use --no-deps on torch 2.x (TabDDPM still works)
%pip install -q ForestDiffusion xgboost category-encoders imbalanced-learn absl-py tensorboardX icecream dython optuna skorch pyarrow tomli tomli-w
%pip install -q "pynvml>=11,<12"
%pip install -q "libzero==0.0.8" "rtdl==0.0.13" --no-deps

import sys
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
REPO_ROOT = NOTEBOOK_DIR.parents[2]
DIFFUSION_PKG = NOTEBOOK_DIR.parent
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm"))
sys.path.insert(0, str(REPO_ROOT / "_vendor" / "tab-ddpm" / "scripts"))
sys.path.insert(0, str(DIFFUSION_PKG))

from diffusion_generators import train_tabddpm, train_forestdiffusion


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from sdv.metadata import SingleTableMetadata
from sdv.single_table import (
    CTGANSynthesizer,
    CopulaGANSynthesizer,
    TVAESynthesizer,
    GaussianCopulaSynthesizer
)

from sdv.evaluation.single_table import evaluate_quality



# ----------------------------------------------------
# Load Dataset
# ----------------------------------------------------

magic_gamma_telescope = fetch_ucirepo(id=159)

# data (as pandas dataframes)
X = magic_gamma_telescope.data.features
y = magic_gamma_telescope.data.targets

# Combine features and target into a single DataFrame
data = pd.concat([X, y], axis=1)

target_col = y.columns[0] # Correctly identify the target column

# metadata
print("Dataset Metadata:")
print(magic_gamma_telescope.metadata)

# variable information
print("\nDataset Variable Information:")
print(magic_gamma_telescope.variables)

# ----------------------------------------------------
# Metadata
# ----------------------------------------------------
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(data) # Use the combined DataFrame

# ----------------------------------------------------
# Experiment Settings
# ----------------------------------------------------
N_SAMPLES = 1000
TEST_SIZE = 0.2
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ----------------------------------------------------
# Storage Containers
# ----------------------------------------------------
scores = {}

synthetic_datasets = {}

quality_results = []


Dataset Metadata:
{'uci_id': 159, 'name': 'MAGIC Gamma Telescope', 'repository_url': 'https://archive.ics.uci.edu/dataset/159/magic+gamma+telescope', 'data_url': 'https://archive.ics.uci.edu/static/public/159/data.csv', 'abstract': 'Data are MC generated to simulate registration of high energy gamma particles in an atmospheric Cherenkov telescope', 'area': 'Physics and Chemistry', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 19020, 'num_features': 10, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2004, 'last_updated': 'Tue Dec 19 2023', 'dataset_doi': '10.24432/C52C8B', 'creators': ['R. Bock'], 'intro_paper': None, 'additional_info': {'summary': "The data are MC generated (see below) to simulate registration of high energy gamma particles in a ground-based atmospheric Cherenkov gamma telescope using the imaging techniq

In [3]:
# SINGLE RUN

seed = SEED

print("\n================ SINGLE RUN ================")

np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)


# TRAIN / TEST SPLIT (NO LEAKAGE)

train_real, test_real = train_test_split(
    data,
    test_size=TEST_SIZE,
    stratify=data[target_col],
    random_state=seed
)

train_metadata = SingleTableMetadata()
train_metadata.detect_from_dataframe(train_real)

# ---------------------------------------------------
# TabDDPM
# ---------------------------------------------------

try:

    print("Training TabDDPM...")
    synthetic_tabddpm = train_tabddpm(
        train_real,
        target_col=target_col,
        categorical_columns=[target_col],
        n_samples=N_SAMPLES,
        seed=seed,
    )

    synthetic_datasets["TabDDPM"] = synthetic_tabddpm.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_tabddpm,
        metadata=train_metadata,
    )

    scores["TabDDPM"] = quality.get_score()

    print("TabDDPM:", round(scores["TabDDPM"], 4))

except Exception as e:
    print("TabDDPM Failed:", e)



================ SINGLE RUN ================
Training TabDDPM...
[0]
12
{'num_classes': 2, 'is_y_cond': False, 'rtdl_params': {'d_layers': [256, 256, 256], 'dropout': 0.0}, 'd_in': np.int64(12)}
mlp
Step 500/1000 MLoss: 0.0 GLoss: 0.3247 Sum: 0.3247
Step 1000/1000 MLoss: 0.0 GLoss: 0.3134 Sum: 0.3134
mlp
Sample timestep    0
Discrete cols: []
Num shape:  (1000, 10)
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 618.12it/s]|
Column Shapes Score: 94.9%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 302.56it/s]|
Column Pair Trends Score: 87.45%

Overall Score (Average): 91.17%

TabDDPM: 0.9117


In [4]:
# ForestDiffusion

try:
    import traceback

    print("Training ForestDiffusion...")
    synthetic_forestdiffusion = train_forestdiffusion(
        train_real,
        target_col=target_col,
        categorical_columns=[target_col],
        n_samples=N_SAMPLES,
        seed=seed,
    )

    synthetic_datasets["ForestDiffusion"] = synthetic_forestdiffusion.copy()

    quality = evaluate_quality(
        real_data=train_real,
        synthetic_data=synthetic_forestdiffusion,
        metadata=train_metadata,
    )

    scores["ForestDiffusion"] = quality.get_score()

    print("ForestDiffusion:", round(scores["ForestDiffusion"], 4))

    del synthetic_forestdiffusion

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

except Exception as e:

    print("ForestDiffusion Failed:")
    traceback.print_exc()


Training ForestDiffusion...
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 283.95it/s]|
Column Shapes Score: 96.83%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 138.16it/s]|
Column Pair Trends Score: 96.38%

Overall Score (Average): 96.6%

ForestDiffusion: 0.966


In [5]:
# SDV MODELS

sdv_models = {
    "CTGAN": CTGANSynthesizer(metadata=train_metadata),
    "CopulaGAN": CopulaGANSynthesizer(metadata=train_metadata),
    "TVAE": TVAESynthesizer(metadata=train_metadata),
    "GaussianCopula": GaussianCopulaSynthesizer(metadata=train_metadata)
}

for model_name, model in sdv_models.items():

    try:

        model.fit(train_real)

        synthetic_data = model.sample(N_SAMPLES)

        synthetic_datasets[model_name] = synthetic_data.copy()

        quality = evaluate_quality(
            real_data=train_real,
            synthetic_data=synthetic_data,
            metadata=train_metadata
        )

        scores[model_name] = quality.get_score()

        print(
            f"{model_name}: {round(scores[model_name], 4)}"
        )

    except Exception as e:

        print(
            f"{model_name} Failed: {e}"
        )

Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 386.79it/s]|
Column Shapes Score: 88.78%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 195.15it/s]|
Column Pair Trends Score: 89.5%

Overall Score (Average): 89.14%

CTGAN: 0.8914
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 521.20it/s]|
Column Shapes Score: 88.66%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 222.77it/s]|
Column Pair Trends Score: 87.41%

Overall Score (Average): 88.03%

CopulaGAN: 0.8803
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 394.92it/s]|
Column Shapes Score: 90.44%

(2/2) Evaluating Column Pair Trends: |██████████| 55/55 [00:00<00:00, 227.84it/s]|
Column Pair Trends Score: 91.6%

Overall Score (Average): 91.02%

TVAE: 0.9102
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 11/11 [00:00<00:00, 336.51it/s]|
Column Shapes Sc

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier


models = {

    'LogReg': LogisticRegression(max_iter=5000, solver='liblinear', random_state=42),
    'SVM-RBF': LinearSVC(max_iter=2000, dual='auto', random_state=42),
    'KNN': KNeighborsClassifier(),
    'NaiveBayes': GaussianNB(),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42),
    'ExtraTrees':  ExtraTreesClassifier(random_state=42),
    'GradientBoost': GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "MLP": MLPClassifier(max_iter=2000, random_state=42),
}


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.base import clone
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd
import numpy as np

def evaluate_models(
    train_df,
    test_df,
    label_col,
    models,
    test_size=0.2,
    seeds=[42,43,44,45,46,47,48,49,50,51]
):

    results = []

    for name, model in models.items():

        accuracy_scores = []
        f1_scores = []
        precision_scores = []
        recall_scores = []

        for seed in seeds:

            X_train = train_df.drop(columns=[label_col])
            y_train = train_df[label_col]

            X_train, _, y_train, _ = train_test_split(
                X_train,
                y_train,
                test_size=test_size,
                random_state=seed,
                stratify=y_train
            )

            X_test = test_df.drop(columns=[label_col])
            y_test = test_df[label_col]

            _, X_test, _, y_test = train_test_split(
                X_test,
                y_test,
                test_size=test_size,
                random_state=seed,
                stratify=y_test
            )

            scaler = StandardScaler().fit(X_train)

            X_train_s = scaler.transform(X_train)
            X_test_s = scaler.transform(X_test)

            clf = clone(model)

            if hasattr(clf, "random_state"):
                clf.set_params(random_state=seed)

            clf.fit(X_train_s, y_train)

            y_pred = clf.predict(X_test_s)

            accuracy_scores.append(
                accuracy_score(y_test, y_pred)
            )

            f1_scores.append(
                f1_score(
                    y_test,
                    y_pred,
                    pos_label="g",
                    average="binary",
                    zero_division=0
                )
            )

            precision_scores.append(
                precision_score(
                    y_test,
                    y_pred,
                    pos_label="g",
                    average="binary",
                    zero_division=0
                )
            )

            recall_scores.append(
                recall_score(
                    y_test,
                    y_pred,
                    pos_label="g",
                    average="binary",
                    zero_division=0
                )
            )

        results.append({
            "Model": name,

            "Accuracy Mean": np.mean(accuracy_scores),
            "Accuracy Std": np.std(accuracy_scores),

            "F1 Mean": np.mean(f1_scores),
            "F1 Std": np.std(f1_scores),

            "Precision Mean": np.mean(precision_scores),
            "Precision Std": np.std(precision_scores),

            "Recall Mean": np.mean(recall_scores),
            "Recall Std": np.std(recall_scores),

            "Accuracy (Mean±Std)": f"{np.mean(accuracy_scores):.4f} \u00b1 {np.std(accuracy_scores):.4f}",
            "F1 (Mean±Std)": f"{np.mean(f1_scores):.4f} \u00b1 {np.std(f1_scores):.4f}",
            "Precision (Mean±Std)": f"{np.mean(precision_scores):.4f} \u00b1 {np.std(precision_scores):.4f}",
            "Recall (Mean±Std)": f"{np.mean(recall_scores):.4f} \u00b1 {np.std(recall_scores):.4f}"
        })

    return pd.DataFrame(results).sort_values(
        by="Accuracy Mean",
        ascending=False
    )


In [8]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd

label_col = target_col

model_order = [
    "CTGAN",
    "CopulaGAN",
    "TVAE",
    "GaussianCopula",
    "ForestDiffusion",
    "TabDDPM"
]

seeds = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

print("TRTR (Train Real, Test Real)")

trtr_results = evaluate_models(
    train_df=data,
    test_df=data,
    label_col="class",
    models=models,
    test_size=TEST_SIZE,
    seeds=seeds
)

display(
    trtr_results[
        [
            "Model",
            "Accuracy (Mean±Std)",
            "F1 (Mean±Std)",
            "Precision (Mean±Std)",
            "Recall (Mean±Std)"
        ]
    ]
)

print("=" * 70)

all_comparisons = []

for synth_name in model_order:

    print(f"{synth_name} - TSTR")

    synthetic_train_df = synthetic_datasets[synth_name]

    tstr_results = evaluate_models(
        train_df=synthetic_train_df,
        test_df=data,
        label_col="class",
        models=models,
        test_size=TEST_SIZE,
        seeds=seeds
    )

    display(
        tstr_results[
            [
                "Model",
                "Accuracy (Mean±Std)",
                "F1 (Mean±Std)",
                "Precision (Mean±Std)",
                "Recall (Mean±Std)"
            ]
        ]
    )

    comparison = trtr_results.merge(
        tstr_results,
        on="Model",
        suffixes=["_TRTR", "_TSTR"]
    )

    comparison["Accuracy_Drop"] = (
        comparison["Accuracy Mean_TRTR"]
        - comparison["Accuracy Mean_TSTR"]
    )

    comparison["F1_Drop"] = (
        comparison["F1 Mean_TRTR"]
        - comparison["F1 Mean_TSTR"]
    )

    comparison["Precision_Drop"] = (
        comparison["Precision Mean_TRTR"]
        - comparison["Precision Mean_TSTR"]
    )

    comparison["Recall_Drop"] = (
        comparison["Recall Mean_TRTR"]
        - comparison["Recall Mean_TSTR"]
    )

    comparison["Synthetic_Model"] = synth_name

    print(f"{synth_name} - TRTR vs TSTR")

    display(
        comparison[
            [
                "Synthetic_Model",
                "Model",
                "Accuracy_Drop",
                "F1_Drop",
                "Precision_Drop",
                "Recall_Drop",
                "Accuracy (Mean±Std)_TRTR",
                "Accuracy (Mean±Std)_TSTR"
            ]
        ]
    )

    all_comparisons.append(comparison)

combined_comparison = pd.concat(
    all_comparisons,
    ignore_index=True
)

summary = (
    combined_comparison
    .groupby("Synthetic_Model", as_index=False)
    [["Accuracy_Drop", "F1_Drop", "Precision_Drop", "Recall_Drop"]]
    .mean()
    .sort_values("Accuracy_Drop")
)

print("Average metric drop by synthetic generator (lower is better)")

display(summary)


TRTR (Train Real, Test Real)


,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
5,RandomForest,0.8803 ± 0.0056,0.9106 ± 0.0042,0.8825 ± 0.0041,0.9407 ± 0.0051
6,ExtraTrees,0.8783 ± 0.0062,0.9100 ± 0.0046,0.8740 ± 0.0054,0.9492 ± 0.0053
9,MLP,0.8780 ± 0.0050,0.9089 ± 0.0038,0.8810 ± 0.0056,0.9386 ± 0.0075
7,GradientBoost,0.8715 ± 0.0048,0.9054 ± 0.0034,0.8657 ± 0.0043,0.9490 ± 0.0035
2,KNN,0.8379 ± 0.0045,0.8826 ± 0.0032,0.8318 ± 0.0037,0.9401 ± 0.0043
8,AdaBoost,0.8261 ± 0.0073,0.8699 ± 0.0062,0.8442 ± 0.0063,0.8974 ± 0.0139
4,DecisionTree,0.8164 ± 0.0046,0.8581 ± 0.0034,0.8599 ± 0.0051,0.8564 ± 0.0040
0,LogReg,0.7909 ± 0.0075,0.8477 ± 0.0054,0.8032 ± 0.0059,0.8974 ± 0.0058
1,SVM-RBF,0.7892 ± 0.0076,0.8469 ± 0.0054,0.8004 ± 0.0060,0.8991 ± 0.0057
3,NaiveBayes,0.7257 ± 0.0080,0.8125 ± 0.0053,0.7297 ± 0.0055,0.9165 ± 0.0061


CTGAN - TSTR


,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
6,ExtraTrees,0.7730 ± 0.0063,0.8308 ± 0.0060,0.8036 ± 0.0038,0.8600 ± 0.0142
5,RandomForest,0.7632 ± 0.0075,0.8215 ± 0.0061,0.8032 ± 0.0066,0.8408 ± 0.0114
1,SVM-RBF,0.7548 ± 0.0062,0.8173 ± 0.0050,0.7904 ± 0.0049,0.8462 ± 0.0084
7,GradientBoost,0.7534 ± 0.0108,0.8114 ± 0.0096,0.8044 ± 0.0052,0.8187 ± 0.0158
0,LogReg,0.7511 ± 0.0057,0.8140 ± 0.0048,0.7892 ± 0.0041,0.8405 ± 0.0090
8,AdaBoost,0.7507 ± 0.0078,0.8094 ± 0.0069,0.8024 ± 0.0130,0.8172 ± 0.0207
3,NaiveBayes,0.7346 ± 0.0077,0.8207 ± 0.0048,0.7302 ± 0.0056,0.9369 ± 0.0049
9,MLP,0.7227 ± 0.0121,0.7880 ± 0.0109,0.7810 ± 0.0081,0.7954 ± 0.0199
2,KNN,0.7126 ± 0.0089,0.7869 ± 0.0075,0.7576 ± 0.0046,0.8187 ± 0.0117
4,DecisionTree,0.6821 ± 0.0144,0.7482 ± 0.0139,0.7688 ± 0.0113,0.7291 ± 0.0237


CTGAN - TRTR vs TSTR


...

,Model,Accuracy (Mean±Std),F1 (Mean±Std),Precision (Mean±Std),Recall (Mean±Std)
3,NaiveBayes,0.6586 ± 0.0120,0.7910 ± 0.0066,0.6558 ± 0.0075,0.9963 ± 0.0085
0,LogReg,0.6485 ± 0.0006,0.7867 ± 0.0003,0.6484 ± 0.0004,0.9999 ± 0.0002
1,SVM-RBF,0.6484 ± 0.0005,0.7867 ± 0.0002,0.6484 ± 0.0003,0.9999 ± 0.0003
8,AdaBoost,0.6453 ± 0.0059,0.7821 ± 0.0045,0.6499 ± 0.0039,0.9821 ± 0.0148
6,ExtraTrees,0.6383 ± 0.0109,0.7684 ± 0.0069,0.6570 ± 0.0068,0.9253 ± 0.0113
5,RandomForest,0.6349 ± 0.0074,0.7667 ± 0.0050,0.6544 ± 0.0043,0.9256 ± 0.0100
9,MLP,0.6311 ± 0.0106,0.7518 ± 0.0088,0.6667 ± 0.0071,0.8622 ± 0.0219
7,GradientBoost,0.6282 ± 0.0084,0.7578 ± 0.0056,0.6559 ± 0.0057,0.8974 ± 0.0119
2,KNN,0.6044 ± 0.0100,0.7243 ± 0.0076,0.6605 ± 0.0065,0.8019 ± 0.0128
4,DecisionTree,0.5560 ± 0.0141,0.6603 ± 0.0113,0.6549 ± 0.0106,0.6659 ± 0.0132


TabDDPM - TRTR vs TSTR


,Synthetic_Model,Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop,Accuracy (Mean±Std)_TRTR,Accuracy (Mean±Std)_TSTR
0,TabDDPM,RandomForest,0.245478,0.143949,0.228085,0.015085,0.8803 ± 0.0056,0.6349 ± 0.0074
1,TabDDPM,ExtraTrees,0.240011,0.141687,0.217012,0.023925,0.8783 ± 0.0062,0.6383 ± 0.0109
2,TabDDPM,MLP,0.246898,0.157095,0.214339,0.076440,0.8780 ± 0.0050,0.6311 ± 0.0106
3,TabDDPM,GradientBoost,0.243323,0.147627,0.209842,0.051582,0.8715 ± 0.0048,0.6282 ± 0.0084
4,TabDDPM,KNN,0.233544,0.158292,0.171265,0.138200,0.8379 ± 0.0045,0.6044 ± 0.0100
5,TabDDPM,AdaBoost,0.180783,0.087826,0.194344,-0.084672,0.8261 ± 0.0073,0.6453 ± 0.0059
6,TabDDPM,DecisionTree,0.260463,0.197808,0.204915,0.190592,0.8164 ± 0.0046,0.5560 ± 0.0141
7,TabDDPM,LogReg,0.142429,0.060979,0.154751,-0.102514,0.7909 ± 0.0075,0.6485 ± 0.0006
8,TabDDPM,SVM-RBF,0.140799,0.060229,0.151993,-0.100730,0.7892 ± 0.0076,0.6484 ± 0.0005
9,TabDDPM,NaiveBayes,0.067166,0.021522,0.073822,-0.079765,0.7257 ± 0.0080,0.6586 ± 0.0120


Average metric drop by synthetic generator (lower is better)


,Synthetic_Model,Accuracy_Drop,F1_Drop,Precision_Drop,Recall_Drop
2,ForestDiffusion,0.019940,0.015981,0.008963,0.023763
4,TVAE,0.076930,0.074923,-0.000778,0.150770
3,GaussianCopula,0.088812,0.056814,0.089699,0.013025
0,CTGAN,0.089624,0.070440,0.054155,0.088102
1,CopulaGAN,0.111930,0.113724,-0.002311,0.219189
5,TabDDPM,0.200089,0.117701,0.182037,0.012814


In [9]:
output_file = "TRTR_TSTR_results.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    trtr_results.to_excel(
        writer,
        sheet_name="TRTR_Results",
        index=False
    )

    combined_comparison.to_excel(
        writer,
        sheet_name="All_Comparisons",
        index=False
    )

    summary.to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )

    for synth_name in model_order:
        synth_results = combined_comparison[
            combined_comparison["Synthetic_Model"] == synth_name
        ]

        synth_results.to_excel(
            writer,
            sheet_name=synth_name[:31],
            index=False
        )

print(f"Results saved to: {output_file}")

Results saved to: TRTR_TSTR_results.xlsx
